# Train optional learned text-emoji XLM-R

Select a GPU runtime and run cells in order. This notebook runs real fine-tuning
on your assigned GPU. The packaged 16k/2k/2k bootstrap is synthetic, experimental
and cannot establish human-labelled accuracy. Optional context and sarcasm
supervision are disabled until you supply independently reviewed labels.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, os, subprocess, sys
from google.colab import files
uploaded = files.upload()  # Select xlmr_advanced_training.zip
archives = [name for name in uploaded if name.lower().endswith(".zip")]
assert len(archives) == 1, "Upload exactly one training ZIP"
base = Path("/content").resolve()
PACKAGE = base / "xlmr_advanced_training"
with zipfile.ZipFile(archives[0]) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and len(names) < 1000
    assert sum(item.file_size for item in archive.infolist()) < 128 * 1024**2
    for item in archive.infolist():
        name = item.filename
        assert "\\" not in name and not PurePosixPath(name).is_absolute()
        assert ".." not in PurePosixPath(name).parts
        assert name.startswith("xlmr_advanced_training/")
        assert (base / name).resolve().is_relative_to(PACKAGE.resolve())
        assert (item.external_attr >> 16) & 0o170000 != 0o120000
    manifest = json.loads(archive.read("xlmr_advanced_training/MANIFEST.json"))
    expected = {"xlmr_advanced_training/" + name for name in manifest["sha256"]}
    assert set(names) == expected | {"xlmr_advanced_training/MANIFEST.json"}
    for name, digest in manifest["sha256"].items():
        assert hashlib.sha256(archive.read("xlmr_advanced_training/" + name)).hexdigest() == digest
    archive.extractall(base)
PROJECT = PACKAGE / "source"
os.chdir(PROJECT)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[model]"], check=True)
sys.path.insert(0, str(PROJECT))
print("Verified source/data package. No model weights were included.")


## Choose data and optional reviewed supervision

Use defaults to train text/emoji fusion only. Your own emotion CSV can include
reviewed context and binary sarcasm labels: consult START_HERE.md before enabling
either option. Do not manufacture sarcasm labels from emotion or emoji.


In [ ]:
USE_OWN_DATA = False
ALLOW_SYNTHETIC = not USE_OWN_DATA  # Set True explicitly for your mixed CSV retaining synthetic-origin rows
USE_CONTEXT = False
SARCASM_COLUMN = None  # "sarcasm" only with independent reviewed binary labels
EPOCHS = 3
DATA = PACKAGE / "data/all_splits.csv"
AUDIT = PACKAGE / "data/dataset_audit.json"
if USE_OWN_DATA:
    custom = files.upload()  # one canonical CSV, plus approved audit JSON if required
    csvs = [name for name in custom if name.lower().endswith(".csv")]
    audits = [name for name in custom if name.lower().endswith(".json")]
    assert len(csvs) == 1 and len(audits) <= 1
    DATA = Path(csvs[0]).resolve()
    AUDIT = Path(audits[0]).resolve() if audits else None
else:
    assert not USE_CONTEXT and SARCASM_COLUMN is None, "The synthetic bootstrap has no reviewed context/sarcasm labels"
from google.colab import drive
drive.mount("/content/drive")
STORAGE = Path("/content/drive/MyDrive/CodeMood_Advanced_XLMR")
RUN = STORAGE / "xlmr_fusion_emotion"
RUN.mkdir(parents=True, exist_ok=True)
command = [sys.executable, "-u", "-m", "training.train_advanced", "--data", str(DATA),
           "--output", str(RUN), "--split-manifest", str(RUN / "frozen_split.csv"),
           "--epochs", str(EPOCHS), "--batch-size", "2", "--gradient-accumulation", "8",
           "--max-length", "128", "--learning-rate", "2e-5", "--precision", "auto"]
if AUDIT: command += ["--data-audit", str(AUDIT)]
if ALLOW_SYNTHETIC: command += ["--allow-synthetic"]
if USE_CONTEXT: command += ["--use-context"]
if SARCASM_COLUMN: command += ["--sarcasm-column", SARCASM_COLUMN]
subprocess.run(command + ["--dry-run"], cwd=PROJECT, check=True)
if not USE_OWN_DATA:
    from training.data_workflow import prepare_data
    prepared = prepare_data(DATA, "emotion", "raw", 42, RUN / "frozen_split.csv", AUDIT, "all", True)
    assert [len(part) for part in prepared["parts"]] == [16000, 2000, 2000]


## Train and recover completed epochs

The encoder, seven-emotion head, emoji embeddings and gate are updated. The best
validation macro-F1 checkpoint is retained. Drive holds the newest completed
epoch and safe optimizer/RNG recovery state; no CPU training is attempted.


In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime before training"
print("GPU:", torch.cuda.get_device_name(0))
if (RUN / "latest_recovery.json").is_file():
    command += ["--resume"]
subprocess.run(command, cwd=PROJECT, check=True)


## Inspect the recorded scope and download inference weights

Synthetic-only scores do not establish real-world accuracy. Keep the test split
untouched while making model choices. No improvement over the base checkpoints
is claimed without a matched independent comparison.


In [ ]:
import pandas as pd
metrics = json.loads((RUN / "metrics.json").read_text())
print("Evaluation scope:", metrics["evaluation_scope"])
print("Context trained:", metrics["emotion_metadata"]["trained_with_context"])
print("Supervised sarcasm head:", metrics["emotion_metadata"]["trained_sarcasm_head"])
display(pd.DataFrame(metrics["history"]))
display(pd.DataFrame(metrics["test_by_language"]).T)
display(pd.DataFrame(metrics["test"]["per_class"]).T)
from training.cloud_bundle import export_checkpoint
archive = export_checkpoint(RUN, STORAGE / "xlmr_fusion_emotion.zip")
print("Saved inference ZIP:", archive)
files.download(str(archive))


## Install locally

From the laptop project root:
`python -m training.install_checkpoint "C:/path/to/xlmr_fusion_emotion.zip"`

Refresh the Emotion lab and choose the optional learned fusion profile. Existing
base checkpoints are retained. No trained head or capability is implied by this
source package; availability follows the installed completed checkpoint metadata.
